# HW2/HW3 Starter Notebook: MOST Fit + Ultraspherical Expansion

This notebook is a starter scaffold for:
- **HW2**: fit baseline MOST-style transfer functions from station data
- **HW3**: add an ultraspherical (Gegenbauer) representation in mapped stability space

You can run this with real station data or synthetic fallback data.

## Mathematical Setup

Baseline model (example form):
$$\phi_q(\zeta)=a_q(1-b_q\zeta)^{-1/\lambda_q}$$

Map stability to polynomial space:
$$\xi=\tanh(\alpha_{\xi}\zeta), \quad \zeta=\frac{1}{\alpha_{\xi}}\operatorname{artanh}(\xi)$$

Ultraspherical approximation:
$$\phi_q(\zeta)\approx\sum_{n=0}^{N} c_{n,q} C_n^{(\lambda_*)}(\xi(\zeta))$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass

# SciPy is used for nonlinear fitting and Gegenbauer evaluation
from scipy.optimize import curve_fit
from scipy.special import eval_gegenbauer

np.random.seed(42)

In [ ]:
# ---------- User configuration ----------
CSV_PATH = None  # e.g., '../data/my_station_profile.csv'
TARGET = 'phi_m'  # choose 'phi_m' or 'phi_h' for demo
ALPHA_XI = 0.8   # mapping scale alpha_xi (not von Karman constant)
LAMBDA_STAR = 0.25  # Gegenbauer family parameter
N_CANDIDATES = [2, 4, 6, 8]
TRAIN_FRAC = 0.75

# Required columns if CSV_PATH is provided:
# zeta, phi_m, phi_h

In [ ]:
def make_synthetic_data(n=400):
    zeta = np.linspace(-2.0, 0.95/16.0, n)
    # Synthetic baseline MOST-like truth
    a_m_true, b_m_true, lam_m_true = 1.0, 16.0, 4.0
    a_h_true, b_h_true, lam_h_true = 1.0, 16.0, 2.0

    phi_m = a_m_true * (1.0 - b_m_true * zeta) ** (-1.0 / lam_m_true)
    phi_h = a_h_true * (1.0 - b_h_true * zeta) ** (-1.0 / lam_h_true)

    # Add light heteroscedastic noise to mimic observations
    noise_m = (0.02 + 0.01 * np.abs(zeta)) * np.random.randn(n)
    noise_h = (0.03 + 0.01 * np.abs(zeta)) * np.random.randn(n)

    df = pd.DataFrame({
        'zeta': zeta,
        'phi_m': phi_m * (1.0 + noise_m),
        'phi_h': phi_h * (1.0 + noise_h),
    })
    return df

if CSV_PATH:
    df = pd.read_csv(CSV_PATH)
else:
    df = make_synthetic_data()

for col in ['zeta', 'phi_m', 'phi_h']:
    if col not in df.columns:
        raise ValueError(f'Missing required column: {col}')

df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['zeta', 'phi_m', 'phi_h']).copy()
df.head()

In [ ]:
# ---------- HW2: Baseline MOST fit ----------
def phi_model(zeta, a, b, lam):
    # Domain guard for real-valued branch
    base = 1.0 - b * zeta
    base = np.clip(base, 1e-8, None)
    return a * base ** (-1.0 / lam)

target = TARGET
x = df['zeta'].values
y = df[target].values

idx = np.argsort(x)
x = x[idx]
y = y[idx]

split = int(len(x) * TRAIN_FRAC)
x_train, x_test = x[:split], x[split:]
y_train, y_test = y[:split], y[split:]

p0 = [1.0, 12.0, 3.0]
bounds = ([0.1, 0.1, 0.2], [5.0, 80.0, 20.0])
popt, pcov = curve_fit(phi_model, x_train, y_train, p0=p0, bounds=bounds, maxfev=20000)
a_fit, b_fit, lam_fit = popt

yhat_train_most = phi_model(x_train, *popt)
yhat_test_most = phi_model(x_test, *popt)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

rmse_train_most = rmse(y_train, yhat_train_most)
rmse_test_most = rmse(y_test, yhat_test_most)

print('Baseline MOST fit parameters:')
print(f'  a={a_fit:.4f}, b={b_fit:.4f}, lambda={lam_fit:.4f}')
print(f'  RMSE train={rmse_train_most:.6f}')
print(f'  RMSE test ={rmse_test_most:.6f}')

## HW3: Ultraspherical Expansion on Mapped Coordinate

Build a design matrix using Gegenbauer basis functions:
$$A_{i,n}=C_n^{(\lambda_*)}(\xi_i), \quad \xi_i=\tanh(\alpha_{\xi}\zeta_i)$$
and solve least squares for coefficients $c_n$.

In [ ]:
def xi_map(zeta, alpha_xi):
    return np.tanh(alpha_xi * zeta)

def gegenbauer_design(xi, lam_star, N):
    cols = [eval_gegenbauer(n, lam_star, xi) for n in range(N + 1)]
    return np.column_stack(cols)

def fit_ultraspherical(zeta_train, y_train, zeta_eval, lam_star, alpha_xi, N):
    xi_train = xi_map(zeta_train, alpha_xi)
    A_train = gegenbauer_design(xi_train, lam_star, N)
    coeffs, *_ = np.linalg.lstsq(A_train, y_train, rcond=None)

    xi_eval = xi_map(zeta_eval, alpha_xi)
    A_eval = gegenbauer_design(xi_eval, lam_star, N)
    yhat = A_eval @ coeffs
    return coeffs, yhat

results = []
for N in N_CANDIDATES:
    coeffs, yhat_train = fit_ultraspherical(x_train, y_train, x_train, LAMBDA_STAR, ALPHA_XI, N)
    _, yhat_test = fit_ultraspherical(x_train, y_train, x_test, LAMBDA_STAR, ALPHA_XI, N)
    results.append({
        'N': N,
        'rmse_train': rmse(y_train, yhat_train),
        'rmse_test': rmse(y_test, yhat_test),
        'coeffs': coeffs
    })

res_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'coeffs'} for r in results])
best_idx = int(res_df['rmse_test'].idxmin())
best_N = int(res_df.loc[best_idx, 'N'])
best_coeffs = results[best_idx]['coeffs']

print('Ultraspherical CV summary:')
display(res_df)
print(f'Best N by test RMSE: {best_N}')

In [ ]:
# Recompute predictions with best N for plotting
_, yhat_train_ultra = fit_ultraspherical(x_train, y_train, x_train, LAMBDA_STAR, ALPHA_XI, best_N)
_, yhat_test_ultra = fit_ultraspherical(x_train, y_train, x_test, LAMBDA_STAR, ALPHA_XI, best_N)

rmse_train_ultra = rmse(y_train, yhat_train_ultra)
rmse_test_ultra = rmse(y_test, yhat_test_ultra)

plt.figure(figsize=(10, 5))
plt.scatter(x_train, y_train, s=12, alpha=0.5, label='Train obs')
plt.scatter(x_test, y_test, s=20, alpha=0.8, label='Test obs')

x_all = np.concatenate([x_train, x_test])
x_line = np.linspace(x_all.min(), x_all.max(), 400)
y_line_most = phi_model(x_line, *popt)
_, y_line_ultra = fit_ultraspherical(x_train, y_train, x_line, LAMBDA_STAR, ALPHA_XI, best_N)

plt.plot(x_line, y_line_most, lw=2.0, label='MOST fit')
plt.plot(x_line, y_line_ultra, lw=2.0, label=f'Ultraspherical (N={best_N})')

plt.xlabel('zeta')
plt.ylabel(target)
plt.title(f'HW2/HW3 Comparison for {target}')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print('Final RMSE comparison:')
print(f'  MOST train={rmse_train_most:.6f}, test={rmse_test_most:.6f}')
print(f'  ULTRA train={rmse_train_ultra:.6f}, test={rmse_test_ultra:.6f}')

## Student prompts (submit with notebook)

1. Report fitted MOST parameters and their physical interpretation.
2. Justify selected polynomial order $N$ using test error trends.
3. Discuss when ultraspherical augmentation helps and when it does not.
4. Repeat for both `phi_m` and `phi_h`, and compare mode structure.
5. Optional: fit residual-only ultraspherical model on top of MOST baseline.